In [1]:
!python --version

Python 3.14.4


The system cannot find the path specified.


# ___HRNet OCR___
---------------------

In [ ]:
# OCR here stands for Object Contextual Representations not Optical Character Recognition

In [2]:
import os
import timeit
import argparse

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn

In [3]:
print(torch.__version__)

2.11.0+cpu


In [11]:
#-----------------------------------------------------------
# this is an archaic code base put together in 2019!!!!
#-----------------------------------------------------------

# local imports

from lib.config import config, update_config
from lib.core.function import test, testval
from lib.utils.modelsummary import get_model_summary
from lib.utils.utils import create_logger

from lib.models.seg_hrnet_ocr import get_seg_model

In [5]:
# from the docs;

# the pretrained weight was trained with the Cityscapes dataset
# models are trained and tested with the input size of 512x1024 and 1024x2048 respectively
# if multi-scale testing is used, we adopt scales: 0.5,0.75,1.0,1.25,1.5,1.75.

In [6]:
# the config files are in the experiments/ directory
os.listdir(r"./experiments/cityscapes/")

['seg_hrnet_ocr_w48_trainval_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml',
 'seg_hrnet_ocr_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml',
 'seg_hrnet_ocr_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_16_epoch484_paddle.yaml',
 'seg_hrnet_w48_trainval_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484x2.yaml',
 'seg_hrnet_w48_trainval_ohem_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484x2.yaml',
 'seg_hrnet_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml',
 'seg_hrnet_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484_paddle.yaml',
 'seg_hrnet_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_16_epoch484_paddle.yaml',
 'seg_hrnet_w48_train_ohem_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml']

In [14]:
# the config we need is => HRNetV2-W48 + OCR

parser = argparse.ArgumentParser()

parser.add_argument("--cfg", help="experiment configure file name", type=str, required=True)
parser.add_argument("opts", help="Modify config options using the command-line", default=None, nargs=argparse.REMAINDER);

In [15]:
class captures:
    pass

args_namespace = captures()

parser.parse_args(args=["--cfg", r"./experiments/cityscapes/seg_hrnet_ocr_w48_train_512x1024_sgd_lr1e-2_wd5e-4_bs_12_epoch484.yaml",
                        "DATASET.TEST_SET", r"./data/list/cityscapes/test.lst",
                        "TEST.MODEL_FILE", r"./../bin/hrnet_ocr_cs_trainval_8227_torch11.pth",
                        "TEST.SCALE_LIST", "(0.5,0.75,1.0,1.25,1.5,1.75)",
                        "TEST.FLIP_TEST", "True"], namespace=args_namespace);

In [16]:
[attr for attr in dir(args_namespace) if not attr.startswith("__")] # parsed args as attributes of the class (namespace)

['cfg', 'opts']

In [17]:
update_config(cfg=config, args=args_namespace) # update the defaults

In [18]:
config # interesting

CfgNode({'OUTPUT_DIR': 'output', 'LOG_DIR': 'log', 'GPUS': (0, 1, 2, 3), 'WORKERS': 4, 'PRINT_FREQ': 10, 'AUTO_RESUME': False, 'PIN_MEMORY': True, 'RANK': 0, 'CUDNN': CfgNode({'BENCHMARK': True, 'DETERMINISTIC': False, 'ENABLED': True}), 'MODEL': CfgNode({'NAME': 'seg_hrnet_ocr', 'PRETRAINED': 'pretrained_models/hrnetv2_w48_imagenet_pretrained.pth', 'ALIGN_CORNERS': True, 'NUM_OUTPUTS': 2, 'EXTRA': CfgNode({'FINAL_CONV_KERNEL': 1, 'STAGE1': CfgNode({'NUM_MODULES': 1, 'NUM_RANCHES': 1, 'BLOCK': 'BOTTLENECK', 'NUM_BLOCKS': [4], 'NUM_CHANNELS': [64], 'FUSE_METHOD': 'SUM'}), 'STAGE2': CfgNode({'NUM_MODULES': 1, 'NUM_BRANCHES': 2, 'BLOCK': 'BASIC', 'NUM_BLOCKS': [4, 4], 'NUM_CHANNELS': [48, 96], 'FUSE_METHOD': 'SUM'}), 'STAGE3': CfgNode({'NUM_MODULES': 4, 'NUM_BRANCHES': 3, 'BLOCK': 'BASIC', 'NUM_BLOCKS': [4, 4, 4], 'NUM_CHANNELS': [48, 96, 192], 'FUSE_METHOD': 'SUM'}), 'STAGE4': CfgNode({'NUM_MODULES': 3, 'NUM_BRANCHES': 4, 'BLOCK': 'BASIC', 'NUM_BLOCKS': [4, 4, 4, 4], 'NUM_CHANNELS': [48,

In [19]:
# this is where the actual work happens

# cudnn related setting
cudnn.benchmark = config.CUDNN.BENCHMARK
cudnn.deterministic = config.CUDNN.DETERMINISTIC
cudnn.enabled = config.CUDNN.ENABLED

model = get_seg_model(cfg=config)

RuntimeError: No such file pretrained_models/hrnetv2_w48_imagenet_pretrained.pth

In [ ]:


# model = eval("models." + config.MODEL.NAME + ".get_seg_model")(config)

dump_input = torch.rand((1, 3, config.TRAIN.IMAGE_SIZE[1], config.TRAIN.IMAGE_SIZE[0]))
logger.info(get_model_summary(model.cuda(), dump_input.cuda()))

    if config.TEST.MODEL_FILE:
        model_state_file = config.TEST.MODEL_FILE
    else:
        model_state_file = os.path.join(final_output_dir, "final_state.pth")
    logger.info("=> loading model from {}".format(model_state_file))

    pretrained_dict = torch.load(model_state_file)
    if "state_dict" in pretrained_dict:
        pretrained_dict = pretrained_dict["state_dict"]
    model_dict = model.state_dict()
    pretrained_dict = {k[6:]: v for k, v in pretrained_dict.items() if k[6:] in model_dict.keys()}
    for k, _ in pretrained_dict.items():
        logger.info("=> loading {} from pretrained model".format(k))
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)

    gpus = list(config.GPUS)
    model = nn.DataParallel(model, device_ids=gpus).cuda()

    # prepare data
    test_size = (config.TEST.IMAGE_SIZE[1], config.TEST.IMAGE_SIZE[0])
    test_dataset = eval("datasets." + config.DATASET.DATASET)(
        root=config.DATASET.ROOT,
        list_path=config.DATASET.TEST_SET,
        num_samples=None,
        num_classes=config.DATASET.NUM_CLASSES,
        multi_scale=False,
        flip=False,
        ignore_label=config.TRAIN.IGNORE_LABEL,
        base_size=config.TEST.BASE_SIZE,
        crop_size=test_size,
        downsample_rate=1,
    )

    testloader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=config.WORKERS, pin_memory=True)

    start = timeit.default_timer()
    if "val" in config.DATASET.TEST_SET:
        mean_IoU, IoU_array, pixel_acc, mean_acc = testval(config, test_dataset, testloader, model)

        msg = "MeanIU: {: 4.4f}, Pixel_Acc: {: 4.4f}, \
            Mean_Acc: {: 4.4f}, Class IoU: ".format(mean_IoU, pixel_acc, mean_acc)
        logging.info(msg)
        logging.info(IoU_array)
    elif "test" in config.DATASET.TEST_SET:
        test(config, test_dataset, testloader, model, sv_dir=final_output_dir)


